# OOD 결과 테이블 생성

WandB에서 수집한 OOD 평가 결과를 엑셀 테이블로 생성합니다.

## 📋 실행 순서

### Option A: 이미 CSV 파일이 있는 경우 (권장)
1. **셀 1-6**을 순서대로 실행

### Option B: WandB에서 최신 데이터를 가져오는 경우
1. **셀 0-1, 0-2, 0-3**을 실행하여 WandB에서 데이터 수집
   - ⚠️ **주의**: 시간이 오래 걸릴 수 있습니다 (1-5분)
   - `results/finished_runs_data.csv` 파일이 생성/업데이트됩니다
2. **셀 1-6**을 순서대로 실행

---

## 0. WandB에서 데이터 수집 (선택사항)

**이미 `results/finished_runs_data.csv` 파일이 있다면 이 섹션을 건너뛰고 셀 1부터 시작하세요!**


In [1]:
# 0-1. WandB API 초기화
import wandb
from pathlib import Path

api = wandb.Api()
project_name = "mmea-owcl/Experimental Results on the MMEA-OWCL (Evaluation CL & OOD)"

print(f"📊 연결된 프로젝트: {project_name}")
print("✅ WandB API 초기화 완료")


wandb: Currently logged in as: imhyejeong (hyejeongim) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


📊 연결된 프로젝트: mmea-owcl/Experimental Results on the MMEA-OWCL (Evaluation CL & OOD)
✅ WandB API 초기화 완료


In [2]:
# 0-2. WandB에서 데이터 수집
import pandas as pd
import numpy as np

def get_all_runs_from_project(project_name, filters=None):
    """프로젝트의 모든 Run 데이터를 가져옵니다"""
    api = wandb.Api()
    runs = api.runs(project_name, filters=filters) if filters else api.runs(project_name)
    
    runs_data = []
    print(f"🔍 Run 데이터를 가져오는 중...")
    
    for i, run in enumerate(runs, 1):
        if i % 10 == 0:
            print(f"   진행중: {i} runs 처리...")
        
        run_info = {
            'run_id': run.id,
            'run_name': run.name,
            'state': run.state,
            'sweep_id': run.sweep.id if run.sweep else None,
            'sweep_name': run.sweep.name if run.sweep else None,
            'created_at': run.created_at,
        }
        run_info.update(run.config)
        run_info.update(run.summary._json_dict)
        runs_data.append(run_info)
    
    print(f"✅ 총 {len(runs_data)}개의 Run 데이터 수집 완료")
    return runs_data

# 데이터 수집 실행
print("=" * 100)
print("🔄 WandB에서 데이터 수집 시작")
print("=" * 100)
print("⚠️ 시간이 다소 걸릴 수 있습니다 (run 개수에 따라 1-5분)\n")

all_runs_data = get_all_runs_from_project(project_name)
runs_df = pd.DataFrame(all_runs_data)

print(f"\n" + "=" * 100)
print(f"📊 데이터 수집 완료")
print("=" * 100)
print(f"총 Run 개수: {len(runs_df)}")
print(f"총 컬럼 개수: {len(runs_df.columns)}")


🔄 WandB에서 데이터 수집 시작
⚠️ 시간이 다소 걸릴 수 있습니다 (run 개수에 따라 1-5분)

🔍 Run 데이터를 가져오는 중...
   진행중: 10 runs 처리...
   진행중: 20 runs 처리...
   진행중: 30 runs 처리...
   진행중: 40 runs 처리...
   진행중: 50 runs 처리...
   진행중: 60 runs 처리...
   진행중: 70 runs 처리...
   진행중: 80 runs 처리...
   진행중: 90 runs 처리...
   진행중: 100 runs 처리...
   진행중: 110 runs 처리...
   진행중: 120 runs 처리...
   진행중: 130 runs 처리...
   진행중: 140 runs 처리...
   진행중: 150 runs 처리...
   진행중: 160 runs 처리...
   진행중: 170 runs 처리...
   진행중: 180 runs 처리...
   진행중: 190 runs 처리...
   진행중: 200 runs 처리...
   진행중: 210 runs 처리...
   진행중: 220 runs 처리...
   진행중: 230 runs 처리...
   진행중: 240 runs 처리...
   진행중: 250 runs 처리...
   진행중: 260 runs 처리...
   진행중: 270 runs 처리...
   진행중: 280 runs 처리...
   진행중: 290 runs 처리...
   진행중: 300 runs 처리...
   진행중: 310 runs 처리...
   진행중: 320 runs 처리...
   진행중: 330 runs 처리...
   진행중: 340 runs 처리...
   진행중: 350 runs 처리...
   진행중: 360 runs 처리...
   진행중: 370 runs 처리...
   진행중: 380 runs 처리...
   진행중: 390 runs 처리...
   진행중: 400 runs 처리...
   진행중: 4

In [6]:
# 0-3. CSV 파일로 저장
import os
# 노트북 파일이 있는 디렉토리를 기준으로 경로 설정
notebook_dir = Path(os.getcwd()) if 'notebook' in os.getcwd() else Path(os.getcwd()) / 'notebook'
output_dir = notebook_dir / 'results'
output_dir.mkdir(parents=True, exist_ok=True)

# 전체 데이터 저장
output_file = output_dir / 'all_runs_data.csv'
runs_df.to_csv(output_file, index=False)

# 완료된 run만 필터링하여 저장
finished_runs = runs_df[runs_df['state'] == 'finished']

print("=" * 100)
print("💾 CSV 파일 저장 완료")
print("=" * 100)
print(f"✅ 전체 데이터: {output_file}")
print(f"   - 총 Run 개수: {len(runs_df)}")
print(f"   - 파일 크기: {output_file.stat().st_size / 1024 / 1024:.2f} MB")

if len(finished_runs) > 0:
    finished_output = output_dir / 'finished_runs_data.csv'
    finished_runs.to_csv(finished_output, index=False)
    print(f"\n✅ 완료된 Run 파일: {finished_output}")
    print(f"   - 완료된 Run 개수: {len(finished_runs)}")

print("\n✨ 데이터 수집 및 저장 완료!")
print("👉 이제 아래 셀 1부터 실행하여 OOD 결과 테이블을 생성하세요.")


💾 CSV 파일 저장 완료
✅ 전체 데이터: /workspace/MMEA-OWCL/notebook/results/all_runs_data.csv
   - 총 Run 개수: 889
   - 파일 크기: 5.97 MB

✅ 완료된 Run 파일: /workspace/MMEA-OWCL/notebook/results/finished_runs_data.csv
   - 완료된 Run 개수: 825

✨ 데이터 수집 및 저장 완료!
👉 이제 아래 셀 1부터 실행하여 OOD 결과 테이블을 생성하세요.


---

## 1. OOD 결과 분석 시작

여기서부터는 CSV 파일을 읽어서 OOD 결과 테이블을 생성합니다.


# OOD 결과 테이블 생성

WandB에서 수집한 OOD 평가 결과를 엑셀 테이블로 생성합니다.

## 실행 순서
1. 셀 1: 데이터 로드
2. 셀 2: Sweep Name 파싱  
3. 셀 3: 함수 로드
4. 셀 4: 테이블 생성 및 저장

In [7]:
# 1. 데이터 로드
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

# 노트북 파일이 있는 디렉토리를 기준으로 경로 설정
notebook_dir = Path(os.getcwd()) if 'notebook' in os.getcwd() else Path(os.getcwd()) / 'notebook'
csv_path = notebook_dir / 'results' / 'finished_runs_data.csv'

print(f"📂 데이터 파일 경로: {csv_path}")
print(f"   파일 존재 여부: {csv_path.exists()}\n")

if not csv_path.exists():
    print("⚠️  파일을 찾을 수 없습니다!")
    print("👉 먼저 셀 0-1, 0-2, 0-3을 실행하여 데이터를 수집하세요.")
else:
    df = pd.read_csv(csv_path)
    print(f"✅ 데이터 로드 완료: {len(df)} runs")
    print(f"   • 컬럼 수: {len(df.columns)}")

📂 데이터 파일 경로: /workspace/MMEA-OWCL/notebook/results/finished_runs_data.csv
   파일 존재 여부: True

✅ 데이터 로드 완료: 825 runs
   • 컬럼 수: 2706


/workspace/.local/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3508: DtypeWarning: Columns (3,4,11,467) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
# 2. Sweep Name 파싱
def parse_sweep_name(sweep_name):
    if pd.isna(sweep_name):
        return None, None, None
    pattern = r"Evaluate (.+?) \((.+?)\) Method on (.+?) \(Seeds"
    match = re.search(pattern, sweep_name)
    if match:
        return match.group(1).strip(), match.group(2).strip(), match.group(3).strip()
    return None, None, None

df[['model_name_from_sweep', 'modality_from_sweep', 'dataset_from_sweep']] = df['sweep_name'].apply(
    lambda x: pd.Series(parse_sweep_name(x))
)

print("✅ Sweep Name 파싱 완료")

✅ Sweep Name 파싱 완료


In [12]:
# 3. 테이블 생성 함수
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.cell.cell import MergedCell
from openpyxl.utils import get_column_letter

def create_ood_result_table(df, datasets=['DataEgo', 'UESTC-MMEA-CL'], increments=[2],
                            modality='All', seeds=[1993, 1994, 1995, 1996, 1997],
                            cl_methods=None, ood_methods=None, fusion_type=None):
    """OOD 결과를 표 형태로 생성"""
    if cl_methods is None:
        cl_methods = ['Replay', 'iCaRL', 'EWC', 'LwF']
    if ood_methods is None:
        ood_methods = ['MSP_Baseline', 'MaxLogit_Baseline', 'ODIN_Baseline', 'Entropy_Baseline', 
                       'Energy_Baseline', 'ReAct_Baseline', 'ASH_S_Baseline', 'Scale_Baseline', 'LTS_Baseline']
    
    results = []
    for cl_method in cl_methods:
        for ood_method in ood_methods:
            row_data = {'CL_Method': cl_method, 'OOD_Method': ood_method}
            
            for dataset in datasets:
                for increment in increments:
                    filtered = df[
                        (df['model_name_from_sweep'].str.contains(cl_method, na=False)) &
                        (df['modality_from_sweep'] == modality) &
                        (df['dataset_from_sweep'] == dataset)
                    ]
                    if 'increment' in df.columns:
                        filtered = filtered[filtered['increment'] == increment]
                    if 'seed' in df.columns:
                        filtered = filtered[filtered['seed'].isin(seeds)]
                    if fusion_type is not None and 'fusion_type' in df.columns:
                        filtered = filtered[filtered['fusion_type'] == fusion_type]
                    
                    prefix = f"{dataset}_{increment}"
                    
                    # ACC
                    acc_col = 'Task/avg_acc'
                    row_data[f'{prefix}_ACC'] = filtered[acc_col].mean() if acc_col in filtered.columns and len(filtered) > 0 else np.nan
                    
                    # OOD 메트릭 (Average_OOD 패턴)
                    for metric_name, metric_key in [('AUROC', 'auroc'), ('AUPR_ID', 'aupr_id'), 
                                                     ('AUPR_OOD', 'aupr_ood'), ('FPR_95', 'fpr95')]:
                        col = f'Average_OOD/{ood_method}_{metric_key}'
                        row_data[f'{prefix}_{metric_name}'] = filtered[col].mean() if col in filtered.columns and len(filtered) > 0 else np.nan
            
            results.append(row_data)
    
    result_df = pd.DataFrame(results)
    
    # Multi-index columns 생성
    new_columns = []
    for col in result_df.columns:
        if col in ['CL_Method', 'OOD_Method']:
            new_columns.append(('', col))
        else:
            parts = col.rsplit('_', 1)
            if len(parts) == 2:
                dataset_inc, metric = parts[0], parts[1]
                ds_parts = dataset_inc.rsplit('_', 1)
                if len(ds_parts) == 2:
                    new_columns.append((f'{ds_parts[0]} (increment: {ds_parts[1]})', metric))
                else:
                    new_columns.append((dataset_inc, metric))
            else:
                new_columns.append(('', col))
    
    result_df.columns = pd.MultiIndex.from_tuples(new_columns)
    return result_df


def save_ood_table_to_excel(result_df, filename, apply_formatting=True):
    """OOD 결과 테이블을 엑셀로 저장 (포맷팅 포함)"""
    result_df.to_excel(filename, merge_cells=True)
    
    if apply_formatting:
        wb = openpyxl.load_workbook(filename)
        ws = wb.active
        
        # 스타일
        header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
        header_font = Font(bold=True, color="FFFFFF", size=11)
        subheader_fill = PatternFill(start_color="5B9BD5", end_color="5B9BD5", fill_type="solid")
        subheader_font = Font(bold=True, color="FFFFFF", size=10)
        method_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
        method_font = Font(bold=True, size=10)
        center = Alignment(horizontal="center", vertical="center")
        border = Border(left=Side(style='thin'), right=Side(style='thin'),
                       top=Side(style='thin'), bottom=Side(style='thin'))
        
        # 헤더 스타일링
        for row in ws.iter_rows(min_row=1, max_row=2):
            for cell in row:
                if isinstance(cell, MergedCell):
                    continue
                cell.fill = header_fill if cell.row == 1 else subheader_fill
                cell.font = header_font if cell.row == 1 else subheader_font
                cell.alignment = center
                cell.border = border
        
        # CL Method 컬럼
        for cell in ws['B']:
            if not isinstance(cell, MergedCell) and cell.row > 2:
                cell.fill = method_fill
                cell.font = method_font
                cell.alignment = center
                cell.border = border
        
        # OOD Method 및 데이터 셀
        for row in ws.iter_rows(min_row=3):
            for cell in row:
                if isinstance(cell, MergedCell):
                    continue
                cell.alignment = center
                cell.border = border
                if cell.column > 3 and isinstance(cell.value, (int, float)):
                    cell.number_format = '0.0000'
        
        # 열 너비
        ws.column_dimensions['A'].width = 5
        ws.column_dimensions['B'].width = 12
        ws.column_dimensions['C'].width = 18
        for col_idx in range(4, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(col_idx)].width = 12
        
        wb.save(filename)
    
    print(f"✅ 엑셀 저장 완료: {filename}")
    print(f"   • {result_df.shape[0]} rows × {result_df.shape[1]} columns")

print("✅ 함수 로드 완료")


✅ 함수 로드 완료


In [15]:
# 4. 테이블 생성 및 저장

# 설정
target_datasets = ['DataEgo'] # 'UESTC-MMEA-CL', 'DataEgo'
target_increments = [10, 5, 2]  # 각 increment에 대해 별도 파일 생성
target_modality = 'All'
target_seeds = [1993, 1994, 1995, 1996, 1997]
target_fusion_types = ['auxiliary_head_v2_7']  # 'concat', 'auxiliary_head_v2_7'

# 출력 디렉토리 설정
output_dir = notebook_dir / 'results'
output_dir.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("🔄 OOD 결과 테이블 생성 시작")
print("=" * 100)
print(f"\n전체 설정:")
print(f"  • 데이터셋: {target_datasets}")
print(f"  • Increment: {target_increments}")
print(f"  • 모달리티: {target_modality}")
print(f"  • Fusion Types: {target_fusion_types}")
print(f"  • 시드: {target_seeds}")
print(f"\n각 increment와 fusion_type에 대해 별도의 엑셀 파일을 생성합니다.\n")

# 각 fusion_type과 increment에 대해 반복
for fusion_type in target_fusion_types:
    for increment in target_increments:
        print("\n" + "=" * 100)
        print(f"📊 Fusion Type: {fusion_type}, Increment {increment} 처리 중...")
        print("=" * 100)
        
        # 동적 파일명 생성 (fusion_type에 따라 버전 접두사 추가)
        version_prefix = 'v2' if fusion_type == 'auxiliary_head_v2_7' else 'v1'
        dataset_str = '-'.join(target_datasets)
        output_filename = output_dir / f'{version_prefix}_ood_results_{dataset_str}_inc{increment}_{target_modality}_{fusion_type}.xlsx'
        
        print(f"\n현재 설정:")
        print(f"  • Fusion Type: {fusion_type}")
        print(f"  • Increment: {increment}")
        print(f"  • 출력: {output_filename}")
        
        # 테이블 생성
        ood_table = create_ood_result_table(
            df=df,
            datasets=target_datasets,
            increments=[increment],  # 단일 increment
            modality=target_modality,
            seeds=target_seeds,
            fusion_type=fusion_type  # fusion_type 필터링 추가
        )
        
        print(f"\n✅ 테이블 생성 완료: {ood_table.shape[0]} rows × {ood_table.shape[1]} columns")
        
        # 미리보기
        print("\n📋 테이블 미리보기 (처음 5행)")
        print("-" * 100)
        display(ood_table.head(5))
        
        # 엑셀 저장
        print("\n💾 엑셀 파일로 저장 중...")
        save_ood_table_to_excel(ood_table, str(output_filename), apply_formatting=True)
        
        print(f"✅ 완료! 파일: {output_filename}")

print("\n" + "=" * 100)
print(f"🎉 모든 작업 완료! 총 {len(target_fusion_types) * len(target_increments)}개의 엑셀 파일이 생성되었습니다.")
print("=" * 100)


🔄 OOD 결과 테이블 생성 시작

전체 설정:
  • 데이터셋: ['DataEgo']
  • Increment: [10, 5, 2]
  • 모달리티: All
  • Fusion Types: ['auxiliary_head_v2_7']
  • 시드: [1993, 1994, 1995, 1996, 1997]

각 increment와 fusion_type에 대해 별도의 엑셀 파일을 생성합니다.


📊 Fusion Type: auxiliary_head_v2_7, Increment 10 처리 중...

현재 설정:
  • Fusion Type: auxiliary_head_v2_7
  • Increment: 10
  • 출력: /workspace/MMEA-OWCL/notebook/results/v2_ood_results_DataEgo_inc10_All_auxiliary_head_v2_7.xlsx

✅ 테이블 생성 완료: 36 rows × 7 columns

📋 테이블 미리보기 (처음 5행)
----------------------------------------------------------------------------------------------------


DataEgo (increment: 10)             \
  CL_Method         OOD_Method                     ACC      AUROC   
0    Replay       MSP_Baseline                  89.588  80.921469   
1    Replay  MaxLogit_Baseline                  89.588  81.387260   
2    Replay      ODIN_Baseline                  89.588  85.096731   
3    Replay   Entropy_Baseline                  89.588  81.633300   
4    Replay    Energy_Baseline                  89.588  81.309066   

  DataEgo_10 (increment: AUPR)            DataEgo_10 (increment: FPR)  
                            ID        OOD                          95  
0                    82.928689  80.461396                   73.798450  
1                    83.156837  79.960695                   73.643411  
2                    85.611553  86.563955                   57.674419  
3                    83.651558  82.323932                   68.372093  
4                    83.136827  79.516293                   74.573643


💾 엑셀 파일로 저장 중...
✅ 엑셀 저장 완료: /workspace/MMEA-OWCL/notebook/results/v2_ood_results_DataEgo_inc10_All_auxiliary_head_v2_7.xlsx
   • 36 rows × 7 columns
✅ 완료! 파일: /workspace/MMEA-OWCL/notebook/results/v2_ood_results_DataEgo_inc10_All_auxiliary_head_v2_7.xlsx

📊 Fusion Type: auxiliary_head_v2_7, Increment 5 처리 중...

현재 설정:
  • Fusion Type: auxiliary_head_v2_7
  • Increment: 5
  • 출력: /workspace/MMEA-OWCL/notebook/results/v2_ood_results_DataEgo_inc5_All_auxiliary_head_v2_7.xlsx

✅ 테이블 생성 완료: 36 rows × 7 columns

📋 테이블 미리보기 (처음 5행)
----------------------------------------------------------------------------------------------------


DataEgo (increment: 5)             \
  CL_Method         OOD_Method                    ACC      AUROC   
0    Replay       MSP_Baseline                 87.378  75.295473   
1    Replay  MaxLogit_Baseline                 87.378  76.635823   
2    Replay      ODIN_Baseline                 87.378  77.181030   
3    Replay   Entropy_Baseline                 87.378  75.716440   
4    Replay    Energy_Baseline                 87.378  76.623668   

  DataEgo_5 (increment: AUPR)            DataEgo_5 (increment: FPR)  
                           ID        OOD                         95  
0                   74.694005  67.168753                  78.711768  
1                   77.595591  66.355837                  78.879447  
2                   77.409887  70.914863                  72.118581  
3                   75.524061  68.425872                  76.965227  
4                   77.597857  65.975457                  80.202891


💾 엑셀 파일로 저장 중...
✅ 엑셀 저장 완료: /workspace/MMEA-OWCL/notebook/results/v2_ood_results_DataEgo_inc5_All_auxiliary_head_v2_7.xlsx
   • 36 rows × 7 columns
✅ 완료! 파일: /workspace/MMEA-OWCL/notebook/results/v2_ood_results_DataEgo_inc5_All_auxiliary_head_v2_7.xlsx

📊 Fusion Type: auxiliary_head_v2_7, Increment 2 처리 중...

현재 설정:
  • Fusion Type: auxiliary_head_v2_7
  • Increment: 2
  • 출력: /workspace/MMEA-OWCL/notebook/results/v2_ood_results_DataEgo_inc2_All_auxiliary_head_v2_7.xlsx

✅ 테이블 생성 완료: 36 rows × 7 columns

📋 테이블 미리보기 (처음 5행)
----------------------------------------------------------------------------------------------------


DataEgo (increment: 2)             \
  CL_Method         OOD_Method                    ACC      AUROC   
0    Replay       MSP_Baseline                  89.34  73.577164   
1    Replay  MaxLogit_Baseline                  89.34  72.534233   
2    Replay      ODIN_Baseline                  89.34  74.264987   
3    Replay   Entropy_Baseline                  89.34  73.900507   
4    Replay    Energy_Baseline                  89.34  72.374833   

  DataEgo_2 (increment: AUPR)            DataEgo_2 (increment: FPR)  
                           ID        OOD                         95  
0                   70.109403  65.545818                  77.250247  
1                   71.574932  61.938030                  80.828502  
2                   72.704111  67.555137                  72.139744  
3                   70.909175  66.770698                  75.651729  
4                   71.540322  61.425606                  82.293570


💾 엑셀 파일로 저장 중...
✅ 엑셀 저장 완료: /workspace/MMEA-OWCL/notebook/results/v2_ood_results_DataEgo_inc2_All_auxiliary_head_v2_7.xlsx
   • 36 rows × 7 columns
✅ 완료! 파일: /workspace/MMEA-OWCL/notebook/results/v2_ood_results_DataEgo_inc2_All_auxiliary_head_v2_7.xlsx

🎉 모든 작업 완료! 총 3개의 엑셀 파일이 생성되었습니다.


---

## 📚 사용법 및 커스터마이징

### ✅ 기본 사용법

위의 4개 셀을 순서대로 실행하면 자동으로 엑셀 파일이 생성됩니다.

### 🎯 설정 변경

**셀 4**에서 다음 변수들을 수정하여 원하는 조건으로 테이블을 생성할 수 있습니다:

```python
# 특정 데이터셋만 선택
target_datasets = ['DataEgo']  

# 여러 increment 포함
target_increments = [2, 4, 8]

# 다른 모달리티 선택
target_modality = 'RGB'  # 또는 'Acce', 'Gyro' 등

# fusion_type 선택
target_fusion_types = ['concat', 'auxiliary_head_v2_7']  # 또는 하나만 선택

# 일부 시드만 사용
target_seeds = [1993, 1994]
```

### 🔧 특정 메소드만 선택

함수 호출 시 파라미터를 추가하세요:

```python
ood_table = create_ood_result_table(
    df=df,
    datasets=target_datasets,
    increments=target_increments,
    modality=target_modality,
    seeds=target_seeds,
    fusion_type='concat',                     # 특정 fusion_type만
    cl_methods=['Replay', 'iCaRL'],           # 특정 CL 메소드만
    ood_methods=['Energy_Baseline', 'MSP_Baseline']  # 특정 OOD 메소드만
)
```

### 📊 메트릭 계산 방식

각 메트릭은 `Average_OOD/{method}_{metric}` 컬럼을 사용합니다:
- **AUROC**: `Average_OOD/Energy_Baseline_auroc`
- **AUPR_ID**: `Average_OOD/Energy_Baseline_aupr_id`
- **AUPR_OOD**: `Average_OOD/Energy_Baseline_aupr_ood`
- **FPR_95**: `Average_OOD/Energy_Baseline_fpr95`

이 값들은 1개 seed의 모든 task 평균이므로, 5개 seed에 대해 평균을 내면 최종 결과가 됩니다.

---

## 💾 백업

원본 노트북은 `1.OOD.backup.ipynb`에 백업되어 있습니다.


In [11]:
# # 2. Sweep Name 파싱
# def parse_sweep_name(sweep_name):
#     if pd.isna(sweep_name):
#         return None, None, None
#     pattern = r"Evaluate (.+?) \((.+?)\) Method on (.+?) \(Seeds"
#     match = re.search(pattern, sweep_name)
#     if match:
#         return match.group(1).strip(), match.group(2).strip(), match.group(3).strip()
#     return None, None, None

# df[['model_name_from_sweep', 'modality_from_sweep', 'dataset_from_sweep']] = df['sweep_name'].apply(
#     lambda x: pd.Series(parse_sweep_name(x))
# )

# print("✅ Sweep Name 파싱 완료")